# 04 — Deploy SageMaker Endpoint

Deploy the GST Entity Matcher as a SageMaker real-time endpoint on MAESTRO.

**Run this on MAESTRO SageMaker JupyterLab.**  
Prerequisite: Run `02_run_indexing.ipynb` first (FAISS index must exist in S3).

After deployment, the endpoint URL is what you set as `SAGEMAKER_ENDPOINT_URL` on Airbase.

## 1. Setup

In [ ]:
import os
import sys

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import boto3
import sagemaker
from sagemaker.model import Model

from config import S3_BUCKET, S3_EMBEDDINGS_PREFIX

session = sagemaker.Session()
role = sagemaker.get_execution_role()

print(f"SageMaker role: {role}")
print(f"Region: {session.boto_region_name}")

## 2. Package model artifacts

Create `model.tar.gz` containing the inference code and upload to S3.

In [ ]:
# Run from project root
os.chdir(PROJECT_ROOT)

from endpoint.package_model import package_model

package_model()

MODEL_DATA_URL = f"s3://{S3_BUCKET}/{S3_EMBEDDINGS_PREFIX}model.tar.gz"
print(f"Model artifact: {MODEL_DATA_URL}")

## 3. Create SageMaker Model

**NOTE:** The `image_uri` below uses a generic SageMaker PyTorch inference container.
Your MAESTRO environment may use a custom container image — check with your MAESTRO admin
and update accordingly.

Common alternatives:
- MAESTRO may provide a pre-built inference image
- You may need to use `sagemaker.image_uris.retrieve()` with MAESTRO-specific parameters

In [ ]:
# Retrieve the appropriate container image for your MAESTRO environment.
# This uses the standard SageMaker PyTorch inference image — update if
# your MAESTRO setup uses a different image.
image_uri = sagemaker.image_uris.retrieve(
    framework="pytorch",
    region=session.boto_region_name,
    version="2.0",
    py_version="py310",
    instance_type="ml.m5.large",
    image_scope="inference",
)
print(f"Image URI: {image_uri}")

model = Model(
    image_uri=image_uri,
    model_data=MODEL_DATA_URL,
    role=role,
    sagemaker_session=session,
    name="gst-entity-matcher",
    env={
        # These are read by config.py inside the container
        "OPENAI_API_KEY": os.getenv("OPENAI_API_KEY", ""),
        "OPENAI_API_BASE": os.getenv("OPENAI_API_BASE", ""),
    },
)
print("Model created.")

## 4. Deploy endpoint

This creates a SageMaker real-time endpoint. It takes a few minutes to spin up.

In [ ]:
ENDPOINT_NAME = "gst-entity-matcher"
INSTANCE_TYPE = "ml.m5.large"  # ~400MB FAISS index fits in 8GB RAM

predictor = model.deploy(
    initial_instance_count=1,
    instance_type=INSTANCE_TYPE,
    endpoint_name=ENDPOINT_NAME,
)

print(f"\nEndpoint deployed: {ENDPOINT_NAME}")
print(f"Endpoint ARN: {predictor.endpoint_arn}")

## 5. Test the endpoint

In [ ]:
import json

runtime = boto3.client("sagemaker-runtime")

test_payload = json.dumps({"entity_names": ["DBS BANK", "SINGAPORE AIRLINES"]})

response = runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType="application/json",
    Accept="application/json",
    Body=test_payload,
)

results = json.loads(response["Body"].read().decode("utf-8"))
print(json.dumps(results, indent=2))

## 6. Get the endpoint URL for Airbase

The Streamlit app on Airbase needs an HTTP URL to call. On MAESTRO, SageMaker
endpoints are typically exposed via an API Gateway.

**Check with your MAESTRO admin** for the API Gateway URL pattern. It typically
looks like:

```
https://<api-gateway-id>.execute-api.<region>.amazonaws.com/<stage>/endpoints/<endpoint-name>/invocations
```

Set this as the `SAGEMAKER_ENDPOINT_URL` environment variable on Airbase.

In [ ]:
print("=" * 60)
print("DEPLOYMENT COMPLETE")
print("=" * 60)
print()
print(f"Endpoint name: {ENDPOINT_NAME}")
print()
print("Next steps:")
print("1. Get the API Gateway URL from your MAESTRO admin")
print("2. Set SAGEMAKER_ENDPOINT_URL on Airbase to that URL")
print("3. Deploy the Streamlit app to Airbase")

## Cleanup (optional)

To delete the endpoint and stop incurring costs, uncomment and run the cell below.

In [ ]:
# Uncomment to delete the endpoint
# predictor.delete_endpoint()
# print(f"Endpoint '{ENDPOINT_NAME}' deleted.")